In [3]:
# ====================================================================================
# FASE 2 - PASSO 1: IMPORTAÇÃO DE LIBS E REMOÇÃO DE DUPLICADAS NAS TABELAS ISOLADAS
# ====================================================================================

In [4]:
import pandas as pd
import numpy as np
import os

In [5]:
# Definindo o caminho da pasta onde estão os arquivos (ajuste se necessário)
# Se os arquivos estiverem na mesma pasta do notebook, deixe apenas ''
DATA_PATH = ''

print("Carregando o núcleo da base de dados...")

Carregando o núcleo da base de dados...


In [6]:
# 1. Tabela Principal de Pedidos
df_orders = pd.read_csv(os.path.join(DATA_PATH, 'olist_orders_dataset.csv'))

In [7]:
# 2. Tabela de Itens do Pedido (Preço e Frete)
df_order_items = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_items_dataset.csv'))

In [8]:
# 3. Tabela de Pagamentos (Faturamento)
df_order_payments = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_payments_dataset.csv'))

In [9]:
print(f"Sucesso! Linhas carregadas:")
print(f"- Pedidos (Orders): {df_orders.shape[0]}")
print(f"- Itens (Items): {df_order_items.shape[0]}")
print(f"- Pagamentos (Payments): {df_order_payments.shape[0]}")

Sucesso! Linhas carregadas:
- Pedidos (Orders): 99441
- Itens (Items): 112650
- Pagamentos (Payments): 103886


In [10]:
print("Verificando duplicadas nas tabelas...")

Verificando duplicadas nas tabelas...


In [11]:
# Armazenando o tamanho inicial para comparação
ord_ant = df_orders.shape[0]
itm_ant = df_order_items.shape[0]
pay_ant = df_order_payments.shape[0]

In [12]:
# Removendo linhas 100% duplicadas (se houver)
df_orders = df_orders.drop_duplicates()
df_order_items = df_order_items.drop_duplicates()
df_order_payments = df_order_payments.drop_duplicates()

In [13]:
print(f"Duplicadas removidas:")
print(f"- Pedidos: Linhas antes: {ord_ant} -> Após remoção: {df_orders.shape[0]} (Removidas: {ord_ant - df_orders.shape[0]})")
print(f"- Itens: Linhas antes: {itm_ant} -> Após remoção: {df_order_items.shape[0]} (Removidas: {itm_ant - df_order_items.shape[0]})")
print(f"- Pagamentos: Linhas antes: {pay_ant} -> Após remoção: {df_order_payments.shape[0]} (Removidas: {pay_ant - df_order_payments.shape[0]})")

Duplicadas removidas:
- Pedidos: Linhas antes: 99441 -> Após remoção: 99441 (Removidas: 0)
- Itens: Linhas antes: 112650 -> Após remoção: 112650 (Removidas: 0)
- Pagamentos: Linhas antes: 103886 -> Após remoção: 103886 (Removidas: 0)


In [14]:
# ====================================================================================
# FASE 2 - PASSO 2: UNIFICAÇÃO DAS TABELAS (MERGE)
# ====================================================================================

In [15]:
print("Iniciando o cruzamento das bases de dados...")

Iniciando o cruzamento das bases de dados...


In [16]:
# 1. Cruzando Pedidos com Itens (Garante a visão de produtos por pedido)
# Usamos 'left' para manter todos os pedidos, mesmo os que não têm itens (ex: cancelados muito no início)
df_consolidado = pd.merge(df_orders, df_order_items, on='order_id', how='left')

In [17]:
# 2. Cruzando o resultado anterior com os Pagamentos (Traz a visão financeira)
df_consolidado = pd.merge(df_consolidado, df_order_payments, on='order_id', how='left')

In [18]:
print("Unificação concluída com sucesso!")
print(f"Dimensões da base final consolidada: {df_consolidado.shape[0]} linhas e {df_consolidado.shape[1]} colunas.")

Unificação concluída com sucesso!
Dimensões da base final consolidada: 118434 linhas e 18 colunas.


In [19]:
# Olhada rápida nas primeiras 3 linhas para garantir que as colunas se uniram
df_consolidado.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,1.0,credit_card,1.0,18.12
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,3.0,voucher,1.0,2.00
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,2.0,voucher,1.0,18.59


In [20]:
# ====================================================================================
# FASE 2 - PASSO 3: CONVERSÃO DE DATAS E TRATAMENTO DE NULOS
# ====================================================================================

In [21]:
print("Ajustando a tipagem das datas no df_consolidado...")

Ajustando a tipagem das datas no df_consolidado...


In [22]:
# Lista das colunas de data que vieram da tabela de pedidos
colunas_data_final = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

In [23]:
# Convertendo em lote para datetime
for coluna in colunas_data_final:
    df_consolidado[coluna] = pd.to_datetime(df_consolidado[coluna])

In [24]:
print("Verificando a quantidade de valores nulos por coluna:")
print(df_consolidado.isnull().sum())

Verificando a quantidade de valores nulos por coluna:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 176
order_delivered_carrier_date     2074
order_delivered_customer_date    3397
order_estimated_delivery_date       0
order_item_id                     830
product_id                        830
seller_id                         830
shipping_limit_date               830
price                             830
freight_value                     830
payment_sequential                  3
payment_type                        3
payment_installments                3
payment_value                       3
dtype: int64


In [78]:
# Tratar os vazios da coluna de dias_atraso transformando em 0
df_consolidado['dias_atraso'] = df_consolidado['dias_atraso'].replace('', np.nan).fillna(0).astype(int)

In [79]:
# Tratar as colunas de data: converter para datetime do pandas
# O argumento errors='coerce' transforma automaticamente qualquer texto vazio "" ou inválido em NaT (nulo)
colunas_data = ['data_aprovacao_link', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

for col in colunas_data:
    if col in df_consolidado.columns:
        df_consolidado[col] = pd.to_datetime(df_consolidado[col], errors='coerce')

In [25]:
# ====================================================================================
# FASE 2 - PASSO 4: ENGENHARIA DE RECURSOS (MÉTRICAS)
# ====================================================================================

In [26]:
print("Criando métricas de logística e negócio...")

Criando métricas de logística e negócio...


In [27]:
# 1. Tempo de Entrega Real (Dias)
df_consolidado['tempo_entrega_real_dias'] = (
    df_consolidado['order_delivered_customer_date'] - df_consolidado['order_purchase_timestamp']
).dt.days

In [28]:
# 2. Dias de Atraso (Em relação ao prazo prometido)
df_consolidado['dias_atraso'] = (
    df_consolidado['order_delivered_customer_date'] - df_consolidado['order_estimated_delivery_date']
).dt.days

In [29]:
print("Métricas criadas com sucesso!")

Métricas criadas com sucesso!


In [30]:
# Vamos dar uma olhada em como ficaram as novas colunas para os pedidos entregues
df_consolidado[df_consolidado['order_status'] == 'delivered'][
    ['order_id', 'order_status', 'tempo_entrega_real_dias', 'dias_atraso']
].head(3)

,order_id,order_status,tempo_entrega_real_dias,dias_atraso
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,8.0,-8.0
1,e481f51cbdc54678b7cc49136f2d6af7,delivered,8.0,-8.0
2,e481f51cbdc54678b7cc49136f2d6af7,delivered,8.0,-8.0


In [31]:
# ====================================================================================
# FASE 2 - PASSO 4.2: CÁLCULO DE FATURAMENTO E TICKET MÉDIO
# ====================================================================================

In [32]:
print("Calculando o faturamento total por pedido exclusivo...")

Calculando o faturamento total por pedido exclusivo...


In [33]:
# Agrupando por pedido para descobrir o valor real total gasto em cada um
df_faturamento_pedido = df_consolidado.groupby('order_id')['price'].sum().reset_index()
df_faturamento_pedido.columns = ['order_id', 'valor_total_pedido']

In [34]:
# Trazendo essa informação de volta para o nosso dataframe consolidado
df_consolidado = pd.merge(df_consolidado, df_faturamento_pedido, on='order_id', how='left')

In [35]:
# Calculando o Ticket Médio Geral do E-commerce (Apenas de pedidos aprovados/entregues)
pedidos_validos = df_consolidado[df_consolidado['price'].notnull()]
ticket_medio_geral = pedidos_validos['valor_total_pedido'].unique().mean()

In [36]:
print(f"Métricas Financeiras prontas!")
print(f"-> O Ticket Médio Geral por pedido na Olist é de: R$ {ticket_medio_geral:.2f}")

Métricas Financeiras prontas!
-> O Ticket Médio Geral por pedido na Olist é de: R$ 345.01


In [37]:
# ====================================================================================
# FASE 2 - PASSO 5: EXPORTAÇÃO DA BASE CONSOLIDADA CLEAN
# ====================================================================================

In [38]:
print("Iniciando a exportação do dataset consolidado...")

Iniciando a exportação do dataset consolidado...


In [39]:
# Salvando em CSV na mesma pasta do seu projeto
# Usamos index=False para o Pandas não criar uma coluna de números inúteis no arquivo final
df_consolidado.to_csv('olist_consolidado_clean.csv', index=False)

In [40]:
print("🚀 Sucesso! O arquivo 'olist_consolidado_clean.csv' foi gerado e salvo.")
print(f"Pronto para ser consumido nas próximas fases do projeto!")

🚀 Sucesso! O arquivo 'olist_consolidado_clean.csv' foi gerado e salvo.
Pronto para ser consumido nas próximas fases do projeto!


In [41]:
# ====================================================================================
# FASE 3 - PASSO 1: DASHBOARD ESTATÍSTICO INICIAL (VISÃO GERAL)
# ====================================================================================

In [42]:
# 1. Carregando a nossa base limpa e consolidada
df = pd.read_csv('olist_consolidado_clean.csv')

In [43]:
print("📊 GERANDO DASHBOARD ESTATÍSTICO INICIAL...")

📊 GERANDO DASHBOARD ESTATÍSTICO INICIAL...


In [44]:
# KPIs Principais
faturamento_total = df['price'].sum()
total_pedidos = df['order_id'].nunique()
ticket_medio = faturamento_total / total_pedidos

In [45]:
# Análise de comportamento de compra
itens_por_pedido = df.groupby('order_id')['order_item_id'].max().mean()

In [46]:
print("-" * 50)
print(f"💰 Faturamento Bruto de Itens: R$ {faturamento_total:,.2f}")
print(f"📦 Volume Total de Pedidos Únicos: {total_pedidos:,}")
print(f"💳 Ticket Médio Calculado (Faturamento/Pedidos): R$ {ticket_medio:.2f}")
print(f"🛒 Média de Produtos por Carrinho: {itens_por_pedido:.2f} itens")
print("-" * 50)

--------------------------------------------------
💰 Faturamento Bruto de Itens: R$ 14,209,250.31
📦 Volume Total de Pedidos Únicos: 99,441
💳 Ticket Médio Calculado (Faturamento/Pedidos): R$ 142.89
🛒 Média de Produtos por Carrinho: 1.14 itens
--------------------------------------------------


In [47]:
# ====================================================================================
# FASE 3 - PASSO 2: ANÁLISE DE LOGÍSTICA (PRAZOS E ATRASOS)
# ====================================================================================

In [48]:
print("🚚 ANALISANDO A PERFORMANCE LOGÍSTICA...")

🚚 ANALISANDO A PERFORMANCE LOGÍSTICA...


In [49]:
# Filtrando apenas pedidos que de fato foram entregues para o cliente
df_entregues = df[df['order_status'] == 'delivered'].drop_duplicates('order_id')

In [50]:
media_entrega = df_entregues['tempo_entrega_real_dias'].mean()
atrasados = df_entregues[df_entregues['dias_atraso'] > 0]
taxa_atraso = (len(atrasados) / len(df_entregues)) * 100

In [51]:
print("-" * 50)
print(f"⏱️ Tempo Médio de Entrega: {media_entrega:.1f} dias")
print(f"🚨 Taxa de Pedidos com Atraso: {taxa_atraso:.2f}%")
print(f"✅ Pedidos Entregues no Prazo ou Adiantados: {100 - taxa_atraso:.2f}%")
print("-" * 50)

--------------------------------------------------
⏱️ Tempo Médio de Entrega: 12.1 dias
🚨 Taxa de Pedidos com Atraso: 6.77%
✅ Pedidos Entregues no Prazo ou Adiantados: 93.23%
--------------------------------------------------


In [52]:
# ====================================================================================
# FASE 3 - PASSO 3: ANÁLISE DE CANCELAMENTOS (CHURN DE PEDIDOS)
# ====================================================================================

In [53]:
print("❌ ANALISANDO O IMPACTO DOS CANCELAMENTOS...")

❌ ANALISANDO O IMPACTO DOS CANCELAMENTOS...


In [54]:
# Contando o volume de pedidos por status
status_pedidos = df.drop_duplicates('order_id')['order_status'].value_counts()

total_cancelados = status_pedidos.get('canceled', 0)
taxa_cancelamento = (total_cancelados / total_pedidos) * 100

In [55]:
# Calculando o prejuízo em dinheiro (faturamento perdido com produtos cancelados)
prejuizo_cancelados = df[df['order_status'] == 'canceled']['price'].sum()

In [56]:
print("-" * 50)
print(f"📉 Total de Pedidos Cancelados: {total_cancelados} pedidos")
print(f"🚫 Taxa de Cancelamento Geral: {taxa_cancelamento:.2f}%")
print(f"💸 Faturamento Perdido (Produtos): R$ {prejuizo_cancelados:,.2f}")
print("-" * 50)

--------------------------------------------------
📉 Total de Pedidos Cancelados: 625 pedidos
🚫 Taxa de Cancelamento Geral: 0.63%
💸 Faturamento Perdido (Produtos): R$ 101,340.65
--------------------------------------------------


In [58]:
# ====================================================================================
# FASE 3 - PASSO 3: UNIFICAÇÃO COM ESTADOS (MERGE)
# ====================================================================================

In [59]:
# 1. Carregar a tabela de clientes original
df_customers = pd.read_csv('olist_customers_dataset.csv')

In [60]:
# 2. Selecionar apenas as colunas essenciais para o cruzamento para economizar memória
df_customers_lookup = df_customers[['customer_id', 'customer_state']]

In [61]:
# 3. Fazer o merge com o seu DataFrame consolidado atual (ajuste o nome se o seu df se chamar diferente)
df_consolidado = pd.merge(df_consolidado, df_customers_lookup, on='customer_id', how='left')

In [62]:
# 4. Validar se a coluna de estado foi adicionada com sucesso
print(df_consolidado['customer_state'].value_counts().head())

customer_state
SP    49967
RJ    15420
MG    13738
RS     6521
PR     6017
Name: count, dtype: int64


In [80]:
# Sobrescrevendo o arquivo original com a base tratada e limpa
df_consolidado.to_csv('olist_consolidado_clean.csv', index=False)